## 02_featurization.ipynb

### What is a molecular fingerprint

A molecular fingerprint is a numerical representation of a molecule's
structure. Think of it as a barcode that encodes chemical features:
- Morgan fingerprints: circular substructures around each atom
- MACCS keys: predefined structural features (166 bits)
- RDKit fingerprints: path-based connectivity patterns
- Physicochemical descriptors: MW, LogP, HBD, HBA, etc.
- Mordred descriptors: 1600+ topological, geometric, electronic features

### Imatinib example

Imatinib (Gleevec) SMILES: Cc1ccc(cc1)NC(=O)c2ccc(cc2)F

from src.featurizer import smiles_to_mol, morgan_fingerprint, maccs_keys, rdkit_fingerprint, physicochemical_descriptors
import numpy as np

smiles = r"Cc1ccc(cc1)NC(=O)c2ccc(cc2)F"
mol = smiles_to_mol(smiles)

if mol is not None:
    morgan = morgan_fingerprint(mol)
    maccs = maccs_keys(mol)
    rdkit = rdkit_fingerprint(mol)
    physchem = physicochemical_descriptors(mol)
    
    print(f"Morgan fingerprint shape: {morgan.shape}")
    print(f"MACCS keys shape: {maccs.shape}")
    print(f"RDKit fingerprint shape: {rdkit.shape}")
    print(f"Physicochemical descriptors shape: {physchem.shape}")
    
    # Visualize bits (first 100 bits)
    plt.figure(figsize=(15, 3))
    plt.subplot(1, 4, 1)
    plt.imshow(morgan[:100].reshape(1, -1), aspect='auto', cmap='binary')
    plt.title('Morgan (first 100 bits)')
    
    plt.subplot(1, 4, 2)
    plt.imshow(maccs.reshape(1, -1), aspect='auto', cmap='binary')
    plt.title('MACCS (166 bits)')
    
    plt.subplot(1, 4, 3)
    plt.imshow(rdkit[:100].reshape(1, -1), aspect='auto', cmap='binary')
    plt.title('RDKit (first 100 bits)')
    
    plt.subplot(1, 4, 4)
    plt.bar(range(len(physchem)), physchem)
    plt.title('Physicochemical')
    
    plt.tight_layout()
    plt.show()

### All 6 featurization methods comparison

from src.featurizer import featurize_smiles

methods = ["morgan", "morgan_counts", "maccs", "rdkit", "physchem", "combined", "full"]
for method in methods:
    features = featurize_smiles(smiles, method)
    if features is not None:
        print(f"{method}: shape {features.shape}")

### Feature space similarity comparison

# Tanimoto matrix for 10 known drugs
known_drugs = ["CCO", "CC(=O)OC1=CC=CC=C1C(=O)O", "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
               "CC1=CC=C(C=C1)C(=O)NC2=CC=CC=C2", "CC1=CC=C(C=C1)C(=O)O",
               "CC(C)C1=CC=C(C=C1)C(=O)O", "CC1=CC=C(C=C1)NC(=O)C2=CC=CC=C2",
               "CC1=CC=C(C=C1)C(=O)NC2=CC=CC=C2C(=O)O",
               "CC1=CC=C(C=C1)C(=O)NC2=CC=CC=C2C(=O)NC3=CC=CC=C3",
               "CC1=CC=C(C=C1)C(=O)NC2=CC=CC=C2C(=O)NC3=CC=CC=C3C(=O)O"]

from src.featurizer import compute_similarity_matrix
matrix = compute_similarity_matrix(known_drugs)

import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 8))
sns.heatmap(matrix, annot=True, fmt=".2f", cmap="viridis")
plt.title('Tanimoto Similarity Matrix (Morgan Fingerprints)')
plt.show()

### Featurization speed benchmark

import time
from tqdm import tqdm

# Generate 1000 random SMILES (placeholder)
test_smiles = ["CCO", "CC(=O)OC1=CC=CC=C1C(=O)O"] * 500

methods = ["morgan", "maccs", "rdkit", "physchem", "combined"]
for method in methods:
    start = time.time()
    for smiles in tqdm(test_smiles, desc=f"Method: {method}", leave=False):
        _ = featurize_smiles(smiles, method)
    elapsed = time.time() - start
    print(f"{method}: {elapsed:.2f} seconds ({len(test_smiles)/elapsed:.0f} compounds/sec)")

### Lipinski rule analysis on EGFR dataset

# Load EGFR data and analyze violations
egfr_df = pd.read_csv('data/raw/EGFR kinase_bioactivity.csv')
from src.featurizer import lipinski_violations

violations_list = []
for smiles in egfr_df['canonical_smiles'].head(100):
    mol = smiles_to_mol(smiles)
    if mol is not None:
        violations = lipinski_violations(mol)
        violations_list.append(violations['total'])

import pandas as pd
violation_counts = pd.Series(violations_list).value_counts().sort_index()
print("Lipinski violations distribution:")
print(violation_counts)

plt.figure(figsize=(8, 5))
violation_counts.plot(kind='bar')
plt.title('Lipinski Rule Violations in EGFR Dataset')
plt.xlabel('Number of Violations')
plt.ylabel('Count')
plt.show()